# CharacterTextSplitter

가장 단순한 분할 방식입니다. 기본적으로 `"\n\n"` 을 기준으로 텍스트를 나누고, 청크 크기는 **문자 수**로 측정합니다.

1. 텍스트 분할 방식: 단일 구분자(separator) 기준
2. 청크 크기 측정 방식: 문자 수 기준

> **🔄 최신 버전 기준 변경 사항 (langchain-text-splitters 1.x)**
> - 파일을 읽을 때 `encoding="utf-8"` 을 명시합니다. (Windows 기본 인코딩 cp949에서 한글 파일 읽기 오류 방지)
> - `add_start_index=True` 로 원문 내 청크 시작 위치를 메타데이터에 기록하는 방법을 추가했습니다.
> - 원본의 마지막 셀 `split_documents([file])` 은 **문자열**을 넘겨 오류가 나는 코드였습니다. `Document` 객체를 넘기도록 수정했습니다.

In [ ]:
%pip install -qU langchain-text-splitters

`./data/appendix-keywords.txt` 파일을 읽어 `file` 변수에 저장합니다.

- 🔄 `pathlib.Path.read_text(encoding="utf-8")` 를 사용하면 파일 열기/닫기와 인코딩 지정을 한 줄로 처리할 수 있습니다.

In [ ]:
from pathlib import Path

# 인코딩을 명시하여 파일 전체를 문자열로 읽습니다.
file = Path("./data/appendix-keywords.txt").read_text(encoding="utf-8")

# 앞부분 일부를 출력합니다.
print(file[:500])

## CharacterTextSplitter 생성

- `separator`: 분할 기준 문자열. 기본값은 `"\n\n"`
- `is_separator_regex`: `separator` 를 정규식으로 해석할지 여부 (기본값 `False`)
- `chunk_size`: 청크의 최대 크기 (예: 210자)
- `chunk_overlap`: 인접한 청크 간 중복 크기
- `length_function`: 길이 계산 함수 (기본값 `len`)
- 🔄 `add_start_index`: `True` 이면 각 청크의 원문 내 시작 위치가 `metadata["start_index"]` 에 기록됩니다. 검색 결과를 원문에 하이라이트할 때 유용합니다.

> 참고: `CharacterTextSplitter` 는 구분자로 한 번만 나누기 때문에, 하나의 조각이 `chunk_size` 보다 크면 그대로 남고
> `Created a chunk of size ..., which is longer than the specified ...` 경고가 출력됩니다. 크기를 엄격히 지켜야 한다면 `RecursiveCharacterTextSplitter` 를 사용하세요.

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    separator="\n\n",          # 분할 기준 구분자
    is_separator_regex=False,  # 구분자를 일반 문자열로 취급
    chunk_size=210,            # 청크 최대 크기(문자 수)
    chunk_overlap=0,           # 청크 간 중복 문자 수
    length_function=len,       # 길이 계산 함수
    add_start_index=True,      # 🔄 원문 내 시작 위치를 metadata에 기록
)

`create_documents()` 로 텍스트를 `Document` 리스트로 분할합니다.

In [ ]:
texts = text_splitter.create_documents([file])

print(f"청크 개수: {len(texts)}")
print(f"첫 번째 청크 길이: {len(texts[0].page_content)}")
print(texts[0])  # metadata에 start_index가 포함되어 있습니다.

## 메타데이터와 함께 분할하기

`create_documents()` 에 텍스트 리스트와 같은 길이의 `metadatas` 리스트를 넘기면, 각 원본의 메타데이터가 해당 원본에서 나온 모든 청크에 복사됩니다.

In [ ]:
metadatas = [
    {"document": 1},
    {"document": 2},
]

documents = text_splitter.create_documents(
    [file, file],          # 분할할 텍스트 리스트
    metadatas=metadatas,   # 각 텍스트에 대응하는 메타데이터
)
print(documents[0])

In [ ]:
len(documents)

In [ ]:
# 두 번째 원본(document: 2)에서 나온 마지막 청크의 메타데이터를 확인합니다.
documents[-1].metadata

## split_text()

`Document` 가 아니라 **문자열 리스트**가 필요할 때 사용합니다.

In [ ]:
chunks = text_splitter.split_text(file)
chunks[0]

## split_documents()

🔄 `split_documents()` 는 **`Document` 객체 리스트**를 입력으로 받습니다. (원본 코드처럼 문자열을 넘기면 `AttributeError` 가 발생합니다.)

실제 RAG 파이프라인에서는 문서 로더가 만든 `Document` 리스트를 그대로 넘기는 경우가 대부분입니다.
여기서는 `langchain_core.documents.Document` 로 직접 만듭니다.

> 참고: 과거 예제에서 자주 쓰던 `langchain_community` 의 문서 로더들은 `langchain-community` 패키지가 2026년 5월 지원 종료(sunset)되면서
> 개별 파트너 패키지로 옮겨지는 추세입니다. 단순 텍스트 파일은 위처럼 직접 읽어서 `Document` 를 만드는 것으로 충분합니다.

In [ ]:
from langchain_core.documents import Document

source_docs = [
    Document(
        page_content=file,
        metadata={"source": "./data/appendix-keywords.txt"},
    )
]

split_docs = text_splitter.split_documents(source_docs)
print(f"청크 개수: {len(split_docs)}")
split_docs[0]  # source 메타데이터가 유지되고 start_index가 추가됩니다.